# Notebook 05 workflow definitions

Purpose: reusable launch, monitoring and staged orchestration. Inputs: tagged project definitions and prior Food101 metadata. Running this module top-to-bottom only loads definitions, taking seconds; it creates no experiment or checkpoint. The main `notebooks/05_Learning_and_Resolution_Scaling.ipynb` invokes these definitions. Its exact resume entry is `launch-staged-study`, with the same run ID/profile/source and original clock. Study outputs and checkpoints use `artifacts/notebook_05/<run_id>/` and `checkpoints/<run_id>/stages/`.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat, importlib.util, sys
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)


In [ ]:
"""Notebook05 orchestration. Importing definitions never starts an experiment."""
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shlex, subprocess, sys, time
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file, run_writer
from fmm_jepa.semantic_pilot import configure_environment
from fmm_jepa.food101_training import validate_training_manifest

MAIN_NOTEBOOK = "notebooks/05_Learning_and_Resolution_Scaling.ipynb"
STAGES = [
    {"id": "learning", "image_size": 128, "pretrain_pool": "selected"},
    {"id": "resolution", "image_size": 256, "pretrain_pool": "selected"},
    {"id": "data_volume", "image_size": 128, "pretrain_pool": "full"},
]
PROFILES = {"staged": {"hard_limit_hours": 24, "stage_ids": ["learning", "resolution", "data_volume"]},
            "learning_only": {"hard_limit_hours": 4, "stage_ids": ["learning"]}}
TERMINAL = {"COMPLETED", "FAILED", "INCOMPLETE"}

def _read(path):
    return json.loads(Path(path).read_text())

def _normalized(value):
    return json.loads(json.dumps(value, allow_nan=False))

def _hash(value):
    return hashlib.sha256(json.dumps(_normalized(value), sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def _load(root, name):
    key = "fmm05_" + name
    if key in sys.modules:
        return sys.modules[key]
    path = Path(root) / "experiments/05_scaling" / (name + ".ipynb")
    spec = importlib.util.spec_from_loader(key, TaggedNotebookLoader(path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[key] = module
    try:
        spec.loader.exec_module(module)
    except BaseException:
        sys.modules.pop(key, None)
        raise
    return module

def source_hashes(root):
    root = Path(root)
    result = {}
    for directory in ("fmm_jepa", "experiments/04_followup", "experiments/05_scaling"):
        for path in sorted((root / directory).glob("*.ipynb")):
            cells = ["".join(c["source"]) for c in _read(path)["cells"]
                     if c["cell_type"] == "code" and "ahr-export" in c.get("metadata", {}).get("tags", [])]
            result[str(path.relative_to(root))] = hashlib.sha256("\n".join(cells).encode()).hexdigest()
    return result

def consumer_hash(root):
    return _hash(["".join(c["source"]) for c in _read(Path(root) / MAIN_NOTEBOOK)["cells"] if c["cell_type"] == "code"])

def study_configuration(run_id="learning-resolution-stable-001", profile="staged"):
    if profile not in PROFILES:
        raise ValueError("PROFILE must be staged or learning_only")
    if not isinstance(run_id, str) or not run_id or any(not (x.isalnum() or x in "-_") for x in run_id):
        raise ValueError("RUN_ID must contain only letters, digits, '-' and '_'")
    return {"schema": "fmm-jepa-notebook05-study/v1", "run_id": run_id, "profile": profile,
            **copy.deepcopy(PROFILES[profile]), "device": "cuda:0", "updates": 1000,
            "milestones": [300, 1000], "round_updates": 100, "seeds": [20260928, 20260929, 20260930],
            "ema_start": 0.99, "effective_batch": 128, "probe_epochs": 20,
            "require_learning_gate": True, "evaluation_reserve_seconds": 1800,
            "resolution_evaluation_reserve_seconds": 3600,
            "heldout_enabled": False, "precision": "float32",
            "numerical_backend": "semantic-stable-v1", "max_consecutive_failures": 1,
            "repair_validation_certificate": "artifacts/notebook_05/repair_validation/stability_certificate.json"}

def preview_study(root, config):
    """Read-only estimate. No run directory, deadline, calibration or optimizer is started."""
    if config["profile"] not in PROFILES or config["heldout_enabled"]:
        raise ValueError("Unregistered profile or held-out access")
    if config["updates"] != 1000 or config["milestones"] != [300, 1000]:
        raise ValueError("This notebook registers a 1000-update schedule with 300/1000 snapshots")
    if config["effective_batch"] != 128 or config["probe_epochs"] != 20:
        raise ValueError("Registered batch/probe budget changed")
    root = Path(root)
    old = root / "artifacts/notebook_04/semantic-followup-001"
    split = _read(old / "split_manifest.json")
    training = _read(old / "training_result.json")
    times = {}
    for kind in ("vit", "semantic"):
        values = [a["performance"]["complete_jepa_step_seconds"]["median"]
                  for name, a in training["arms"].items() if ("-" + kind + "-") in name]
        times[kind] = sum(values) / len(values)
    known_training_hours = len(config["seeds"]) * config["updates"] * sum(times.values()) / 3600
    rows = [
        {"stage": "learning", "image_size": 128, "patch_tokens": 256, "pretraining_images": len(split["selected_ids"]["training"]),
         "estimated_hours": [3, 4], "basis": "Measured Notebook04 batch 128 timings; six longer runs plus 18 probes"},
        {"stage": "resolution", "image_size": 256, "patch_tokens": 1024, "pretraining_images": len(split["selected_ids"]["training"]),
         "estimated_hours": [6, 16], "basis": "Unmeasured batch 128 higher-resolution planning range; startup calibration replaces it"},
        {"stage": "data_volume", "image_size": 128, "patch_tokens": 256, "pretraining_images": len(split["pool_ids"]["training"]),
         "estimated_hours": [3, 4], "basis": "Same update count/resolution; larger pretraining pool, fixed probe set"},
    ]
    rows = [r for r in rows if r["stage"] in config["stage_ids"]]
    return {"status": "NOT_RUN", "run_id": config["run_id"], "profile": config["profile"], "stages": rows,
            "estimated_hours": [sum(r["estimated_hours"][0] for r in rows), sum(r["estimated_hours"][1] for r in rows)],
            "hard_limit_hours": config["hard_limit_hours"], "historical128_training_hours_per_stage": known_training_hours,
            "historical128_step_seconds": times, "fixed_probe_training_images": len(split["selected_ids"]["training"]),
            "fixed_development_images": len(split["selected_ids"]["development"]), "source": str(old),
            "learning_gate": "Later stages run only if semantic final-minus-initial paired-seed 95% t lower bound exceeds 0 and representations are finite/nonconstant.",
            "limitations": "Estimate, not a guarantee. Current device load, calibration, data loading and evaluation affect wall time. Source/data verification and all stages share one original deadline."}

def display_preview(preview):
    from IPython.display import Markdown, display
    rows = ["| Stage | Image / tokens | Pretraining images | Estimated time |",
            "|---|---|---:|---:|"]
    for row in preview["stages"]:
        rows.append("| " + row["stage"] + " | " + str(row["image_size"]) + "² / " + str(row["patch_tokens"]) +
                    " | " + format(row["pretraining_images"], ",") + " | " + str(row["estimated_hours"][0]) + "–" + str(row["estimated_hours"][1]) + "h |")
    display(Markdown("\n".join(rows)))
    display(Markdown("**Planning estimate:** " + str(preview["estimated_hours"][0]) + "–" + str(preview["estimated_hours"][1]) +
                     " hours; **hard limit:** " + str(preview["hard_limit_hours"]) + " hours. " + preview["limitations"]))
    return preview

def _status(artifact, stage, status, **details):
    value = {"stage": stage, "status": status, "updated_unix": time.time(), **details}
    atomic_json(Path(artifact) / "study_status.json", value)
    return value

def _assert_binding(root, binding, config):
    if binding["configuration"] != _normalized(config):
        raise ValueError("Study configuration changed; resume requires the original RUN_ID and settings")
    if binding["source_hashes"] != source_hashes(root) or binding["consumer_notebook_sha256"] != consumer_hash(root):
        raise ValueError("Registered definitions or Notebook05 code changed; refusing silent restart")

def _validated_clock(artifact, binding):
    clock = _read(Path(artifact) / "budget_clock.json")
    if clock != binding.get("budget_clock"):
        raise ValueError("Study clock changed from its original registered value")
    maximum = binding["configuration"]["hard_limit_hours"] * 3600
    if clock["max_seconds"] != maximum or clock["deadline_unix"] != clock["started_unix"] + maximum:
        raise ValueError("Original clock does not match the registered profile")
    return clock

def _install_backend(root, study):
    if study.get("numerical_backend") != "semantic-stable-v1" or study.get("max_consecutive_failures") != 1:
        raise ValueError("This repaired study requires semantic-stable-v1 and fail-fast numerical attempts")
    return _load(root, "stable_attention").install()

def _require_repair_validation(root, study):
    """Fail closed until independent numerical and real-checkpoint repair checks pass."""
    path = Path(root) / study["repair_validation_certificate"]
    if not path.is_file():
        raise ValueError("Repair validation certificate is absent; complete the bounded stability checks before launching long training")
    value = _read(path)
    if value.get("schema") != "fmm-jepa-semantic-stability/v1" or value.get("status") != "PASS":
        raise ValueError("Repair validation certificate has not passed")
    required = ("reproduced_failure", "operator_regressions", "late_checkpoint_stress", "exact_resume")
    if any(value.get("checks", {}).get(name, {}).get("status") != "PASS" for name in required):
        raise ValueError("Required repair evidence is incomplete: failure reproduction, operator regression, late-state stress and exact resume must all pass")
    for name in required:
        record = value["checks"][name]
        evidence = Path(record.get("path", ""))
        evidence = evidence if evidence.is_absolute() else Path(root) / evidence
        evidence = evidence.resolve()
        evidence.relative_to(Path(root).resolve())
        if not evidence.is_file() or digest_file(evidence) != record.get("sha256"):
            raise ValueError("Repair evidence file is missing or changed: " + name)
    if value.get("numerical_backend") != study["numerical_backend"]:
        raise ValueError("Repair validation applies to another numerical backend")
    expected = value.get("stable_attention_export_sha256")
    actual = source_hashes(root).get("experiments/05_scaling/stable_attention.ipynb")
    if not expected or expected != actual:
        raise ValueError("Stable attention definitions changed after repair validation")
    return {"path": str(path.resolve()), "sha256": digest_file(path), "status": "PASS", "numerical_backend": study["numerical_backend"]}

def _evaluation_reserve(study, stage):
    return study["resolution_evaluation_reserve_seconds"] if stage["id"] == "resolution" else study["evaluation_reserve_seconds"]

def _prepare_study(root, config):
    root = Path(root).resolve()
    preview = preview_study(root, config)
    repair_validation = _require_repair_validation(root, config)
    artifact = root / "artifacts/notebook_05" / config["run_id"]
    artifact.mkdir(parents=True, exist_ok=True)
    binding_path = artifact / "study_binding.json"
    if binding_path.exists():
        binding = _read(binding_path)
        _assert_binding(root, binding, config)
        if binding.get("repair_validation") != repair_validation:
            raise ValueError("Repair validation certificate changed after launch")
        if not (artifact / "budget_clock.json").exists():
            raise ValueError("Bound run has no clock; cannot reset its deadline")
    else:
        if (artifact / "budget_clock.json").exists() or (artifact / "stages").exists():
            raise ValueError("Unbound pre-existing run evidence; choose a new RUN_ID rather than overwrite")
        now = time.time()
        clock = {"started_unix": now, "deadline_unix": now + config["hard_limit_hours"] * 3600,
                 "max_seconds": config["hard_limit_hours"] * 3600,
                 "policy": "One absolute clock from the first user launch; never reset on restart"}
        binding = {"schema": "notebook05-binding/v1", "configuration": _normalized(config),
                   "source_hashes": source_hashes(root), "consumer_notebook_sha256": consumer_hash(root),
                   "budget_clock": clock, "repair_validation": repair_validation}
        atomic_json(binding_path, binding)
        atomic_json(artifact / "budget_clock.json", clock)
        atomic_json(artifact / "registered_preview.json", preview)
    clock = _validated_clock(artifact, binding)
    return artifact, binding, clock

def _base_config(root, study, stage):
    config = copy.deepcopy(_read(Path(root) / "artifacts/notebook_04/semantic-followup-001/plan.json")["arms"][0]["config"])
    config.update(device=study["device"], dtype="float32", image_size=stage["image_size"])
    config["training"].update(updates=study["updates"], ema_start=study["ema_start"], checkpoint_every=50,
                              microbatch=128, accumulation_steps=1, max_consecutive_failures=study["max_consecutive_failures"])
    config["numerical_backend"] = study["numerical_backend"]
    config["failure_policy"] = "Stop on the first failed numerical attempt; replay only from the last clean checkpoint. Never skip examples."
    config["seeds"].update(model_init=study["seeds"][0], data_order=1101, augmentation=2202, mask=3303, probe=4404)
    config["model"]["feature_seed"] = study["seeds"][0] + 1000
    config["probe"].update(epochs=20, selection="fixed_final_epoch", standardize=True)
    config["evaluation"].update(heldout_enabled=False, batch_size=64, diagnostic_steps=[0, study["updates"]])
    config["budget"].update(max_seconds=study["hard_limit_hours"] * 3600,
                             evaluation_reserve_seconds=_evaluation_reserve(study, stage))
    config["resolution_rationale"] = "Notebook05 controlled " + stage["id"] + "; fixed patch 8, width 192/depth 4 and 1000-update schedule"
    config["comparison"] = "paired-seed equal-update/equal-exposure within-stage; separate fixed-compute curve readout; not parameter-matched"
    return config

def _stage_plan(root, study, stage, config, bundle, calibration, binding):
    micro = int(calibration["selected_microbatch"])
    accumulation = int(calibration["selected_accumulation_steps"])
    if micro * accumulation != study["effective_batch"]:
        raise ValueError("Calibration changed effective batch")
    arms = []
    for i, seed in enumerate(study["seeds"]):
        for kind in ("vit", "semantic"):
            c = copy.deepcopy(config)
            c["training"].update(microbatch=micro, accumulation_steps=accumulation)
            c["evaluation"]["batch_size"] = min(64, micro)
            c["seeds"].update(model_init=seed, data_order=1101 + 10000*i, augmentation=2202 + 10000*i,
                              mask=3303 + 10000*i, probe=4404)
            c["model"]["feature_seed"] = seed + 1000
            arms.append({"id": "s" + str(seed) + "-" + kind, "kind": kind, "seed": seed,
                         "condition": stage["id"], "config": c})
    return _normalized({"schema": "notebook05-stage-plan/v1", "run_id": study["run_id"],
        "stage_id": stage["id"], "stage_role": stage["id"], "stage": stage, "updates": study["updates"],
        "round_updates": study["round_updates"], "milestones": study["milestones"], "seeds": study["seeds"],
        "arms": arms, "train_count": len(bundle["train_ids"]), "probe_train_count": len(bundle["probe_train_ids"]),
        "source_hashes": binding["source_hashes"], "consumer_notebook_sha256": binding["consumer_notebook_sha256"],
        "resume_notebook": MAIN_NOTEBOOK, "resume_cell": "launch-staged-study",
        "evaluation_reserve_seconds": _evaluation_reserve(study, stage), "study_configuration_hash": _hash(study),
        "selection": "Immutable 300/1000-update snapshots from the same1000-update schedule; fixed final probe epoch 20",
        "heldout_status": "NOT_RUN", "numerical_backend": study["numerical_backend"],
        "failure_policy": "Fail immediately on the first numerical attempt; no skipped samples or failed-attempt checkpoints"})

def _remaining_budget(root, artifact, study, stage, calibration, deadline):
    remaining_by_kind = {"vit": 0, "semantic": 0}
    for seed in study["seeds"]:
        for kind in remaining_by_kind:
            path = Path(root) / "checkpoints" / study["run_id"] / "stages" / stage["id"] / "arms" / ("s" + str(seed) + "-" + kind) / "manifest.json"
            completed = int(validate_training_manifest(path.parent)["successful_updates"]) if path.exists() else 0
            if not 0 <= completed <= study["updates"]:
                raise ValueError("Checkpoint step outside registered budget")
            remaining_by_kind[kind] += study["updates"] - completed
    measured = calibration["selected_model_step_seconds"]
    raw = sum(remaining_by_kind[k] * float(measured[k]) for k in remaining_by_kind)
    required = 1.2 * raw + _evaluation_reserve(study, stage) + 300
    evaluation_resume = sum(remaining_by_kind.values()) == 0
    remaining = deadline-time.time()
    result = {"stage": stage["id"], "remaining_updates_by_kind": remaining_by_kind,
              "measured_model_step_seconds": measured, "training_seconds_without_margin": raw,
              "margin": 1.2, "nontraining_reserve_seconds": _evaluation_reserve(study, stage),
              "preparation_reserve_seconds": 300, "required_seconds": required,
              "remaining_seconds": remaining, "fits_budget": remaining > 0 if evaluation_resume else required <= remaining,
              "evaluation_resume_under_original_deadline": evaluation_resume,
              "reserve_is_advisory_for_completed_training": evaluation_resume,
              "update_budget_unchanged": True}
    atomic_json(Path(artifact) / "runtime_estimate.json", result)
    return result

def run_study(root, run_id):
    """Called only by the detached worker launched by Run All."""
    root = Path(root).resolve()
    artifact = root / "artifacts/notebook_05" / run_id
    binding = _read(artifact / "study_binding.json")
    study = binding["configuration"]
    _assert_binding(root, binding, study)
    clock = _validated_clock(artifact, binding)
    deadline = float(clock["deadline_unix"])
    old_status = _read(artifact / "study_status.json") if (artifact / "study_status.json").exists() else {}
    if old_status.get("status") == "COMPLETED":
        return old_status
    if time.time() >= deadline:
        raise TimeoutError("Original study deadline expired; no automatic extension or restart")
    _install_backend(root, study)
    repair_validation = _require_repair_validation(root, study)
    if repair_validation != binding.get("repair_validation"):
        raise ValueError("Repair validation certificate changed after launch")
    preparation, runner, evaluation = (_load(root, x) for x in ("preparation", "runner", "evaluation"))
    stage_results = {}
    with run_writer(artifact):
        try:
            for stage in STAGES:
                role = stage["id"]
                if role not in study["stage_ids"]:
                    stage_results[role] = {"status": "NOT_RUN", "reason": "Not included in selected profile"}
                    continue
                if role != "learning" and study["require_learning_gate"]:
                    gate = stage_results["learning"].get("learning_gate", {})
                    if not gate.get("passed", False):
                        stage_results[role] = {"status": "NOT_RUN", "reason": "Learning stage did not establish positive semantic gain over its own initialization", "learning_gate": gate}
                        continue
                folder = artifact / "stages" / role
                folder.mkdir(parents=True, exist_ok=True)
                (folder / "predictions").mkdir(exist_ok=True)
                complete_path = folder / "workflow_stage_complete.json"
                if complete_path.exists():
                    complete = _read(complete_path)
                    if complete["study_binding_hash"] != _hash(binding) or digest_file(folder / "plan.json") != complete["stage_plan_sha256"]:
                        raise ValueError("Completed stage binding changed")
                    result_path = folder / "evaluation_result.json"
                    if digest_file(result_path) != complete["evaluation_sha256"]:
                        raise ValueError("Completed stage result changed")
                    stage_results[role] = _read(result_path)
                    continue
                _status(artifact, role + ":data_and_calibration", "RUNNING", run_id=run_id)
                config = configure_environment(_base_config(root, study, stage))
                bundle = preparation.prepare_stage_bundle(root, folder, stage, config, verify_images=True)
                calibration = preparation.calibrate_stage(root, folder, stage, config, bundle, deadline)
                atomic_json(folder / "calibration_result.json", calibration)
                if calibration.get("status") not in ("PASS", "COMPLETE", "COMPLETED"):
                    raise RuntimeError("Actual-config calibration did not pass")
                plan = _stage_plan(root, study, stage, config, bundle, calibration, binding)
                path = folder / "plan.json"
                if path.exists() and _read(path) != plan:
                    raise ValueError("Registered stage plan changed")
                if not path.exists():
                    atomic_json(path, plan)
                budget = _remaining_budget(root, folder, study, stage, calibration, deadline)
                if not budget["fits_budget"]:
                    stage_results[role] = {"status": "NOT_RUN", "reason": "Remaining original wall-clock budget cannot fit the registered schedule and evaluation reserve", "runtime_estimate": budget}
                    for later in STAGES[STAGES.index(stage)+1:]:
                        stage_results[later["id"]] = {"status": "NOT_RUN", "reason": "Original budget exhausted before this stage"}
                    report = evaluation.combine_stages(root, artifact, stage_results, plan=study)
                    atomic_json(artifact / "combined_results.json", report)
                    return _status(artifact, role + ":budget", "INCOMPLETE", outcome="INSUFFICIENT_REMAINING_BUDGET",
                                   results=stage_results, elapsed_seconds=time.time()-clock["started_unix"])
                _status(artifact, role + ":resume_preflight", "RUNNING", estimated_remaining_stage_hours=budget["required_seconds"]/3600)
                runner.prepare_stage(root, folder, plan, bundle)
                preflight = runner.verify_stage_resume(root, folder, plan, bundle, deadline)
                atomic_json(folder / "preflight_result.json", preflight)
                _require_repair_validation(root, study)
                _status(artifact, role + ":training", "RUNNING")
                training = runner.train_stage(root, folder, plan, bundle, deadline)
                atomic_json(folder / "training_result.json", training)
                _status(artifact, role + ":evaluation", "RUNNING")
                result = evaluation.evaluate_stage(root, folder, plan, bundle, training, deadline)
                if result.get("status") not in ("PASS", "COMPLETE", "COMPLETED"):
                    raise RuntimeError("Stage evaluation incomplete")
                atomic_json(folder / "evaluation_result.json", result)
                atomic_json(complete_path, {"study_binding_hash": _hash(binding), "stage_plan_sha256": digest_file(path),
                            "evaluation_sha256": digest_file(folder / "evaluation_result.json"), "completed_unix": time.time()})
                stage_results[role] = result
                atomic_json(artifact / "stage_progress.json", {k: {"status": v.get("status"), "learning_gate": v.get("learning_gate")} for k, v in stage_results.items()})
            _assert_binding(root, binding, study)
            report = evaluation.combine_stages(root, artifact, stage_results, plan=study)
            atomic_json(artifact / "combined_results.json", report)
            passed = stage_results.get("learning", {}).get("learning_gate", {}).get("passed", False)
            outcome = "ALL_REQUESTED_STAGES_COMPLETE" if all(stage_results[r].get("status") in ("PASS","COMPLETE","COMPLETED") for r in study["stage_ids"]) else "LEARNING_UNESTABLISHED"
            return _status(artifact, "complete", "COMPLETED", outcome=outcome, learning_gate_passed=passed,
                           stage_statuses={k:v.get("status") for k,v in stage_results.items()},
                           elapsed_seconds=time.time()-clock["started_unix"], heldout_status="NOT_RUN")
        except BaseException as error:
            _status(artifact, "interrupted", "INCOMPLETE" if isinstance(error, TimeoutError) else "FAILED",
                    error=type(error).__name__ + ": " + str(error),
                    numerical_cause=(type(error.__cause__).__name__ + ": " + str(error.__cause__)) if error.__cause__ else None,
                    last_stage=locals().get("role"),
                    elapsed_seconds=time.time()-clock["started_unix"])
            raise

def _session_exists(session):
    return subprocess.run(["tmux", "has-session", "-t", "=" + session], capture_output=True).returncode == 0

def launch_study(root, config):
    """Real Run All launch, with no preview-only toggle and no second writer."""
    root = Path(root).resolve()
    artifact = root / "artifacts/notebook_05" / config["run_id"]
    with run_writer(artifact / "launch_lock"):
        artifact, binding, clock = _prepare_study(root, config)
        session = "fmm-scaling-" + config["run_id"]
        if _session_exists(session):
            _load(root, "deadline_watchdog").launch_watchdog(root, config["run_id"])
            return {"status": "EXISTING_SESSION", "session": session, "artifact": str(artifact), "deadline_unix": clock["deadline_unix"]}
        status = _read(artifact / "study_status.json") if (artifact / "study_status.json").exists() else {}
        if status.get("status") == "COMPLETED":
            return {"status": "COMPLETED", "artifact": str(artifact), "outcome": status.get("outcome")}
        if time.time() >= clock["deadline_unix"]:
            raise TimeoutError("Original deadline expired. Retained evidence is preserved; Run All cannot reset it")
        setup = next(c.source for c in nbformat.read(root / MAIN_NOTEBOOK, as_version=4).cells if c.get("id") == "bootstrap-scaling-study")
        code = setup + "\nWORKFLOW.run_study(ROOT, " + repr(config["run_id"]) + ")\n"
        worker = artifact / "worker.ipynb"
        if worker.exists():
            if [c.source for c in nbformat.read(worker, as_version=4).cells if c.cell_type=="code"] != [code]:
                raise ValueError("Detached worker source changed")
        else:
            nbformat.write(nbformat.v4.new_notebook(cells=[
                nbformat.v4.new_markdown_cell("# Notebook05 detached worker\nFull-state checkpoints: checkpoints/" + config["run_id"] + "/stages/. Resume through the same Notebook05 RUN_ID, profile and unchanged definitions. One persisted absolute deadline."),
                nbformat.v4.new_code_cell(code)], metadata={"kernelspec":{"display_name":"Python (ahr-go)","language":"python","name":"ahr-go"}}), worker)
        command = [sys.executable, "-m", "jupyter", "nbconvert", "--execute", "--to", "notebook", "--inplace",
                   "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=ahr-go", str(worker)]
        shell = "exec " + shlex.join(command) + " > " + shlex.quote(str(artifact / "worker.log")) + " 2>&1"
        _status(artifact, "launching", "STARTING", run_id=config["run_id"])
        launch = {"session":session, "command":command, "artifact":str(artifact), "attach":"tmux attach -t " + session,
                  "detach":"Ctrl-b, then d", "deadline_unix":clock["deadline_unix"], "binding_hash":_hash(binding)}
        atomic_json(artifact / "launch.json", launch)
        try:
            _load(root, "deadline_watchdog").launch_watchdog(root, config["run_id"])
            subprocess.run(["tmux","new-session","-d","-s",session,"-c",str(root),"bash","-c",shell],check=True)
        except BaseException as error:
            _status(artifact, "launch_failed", "FAILED", error=type(error).__name__ + ": " + str(error))
            raise
        return {"status":"LAUNCHED", **launch}

def saved_training_progress(root, run_id, stage_id):
    """Read lightweight manifests only; counts describe durable saved updates."""
    root = Path(root)
    stage_artifact = root / "artifacts/notebook_05" / run_id / "stages" / stage_id
    plan_path = stage_artifact / "plan.json"
    if not plan_path.exists():
        return {"arms": [], "remaining_training_hours": None}
    plan = _read(plan_path)
    calibration_path = stage_artifact / "calibration_result.json"
    times = _read(calibration_path).get("selected_model_step_seconds", {}) if calibration_path.exists() else {}
    rows, forecast = [], 0.0
    for arm in plan["arms"]:
        manifest_path = root / "checkpoints" / run_id / "stages" / stage_id / "arms" / arm["id"] / "manifest.json"
        manifest = _read(manifest_path) if manifest_path.exists() else {}
        completed = int(manifest.get("successful_updates", 0))
        attempted = int(manifest.get("attempted_updates", 0))
        if not 0 <= completed <= plan["updates"] or attempted < completed:
            raise ValueError("Saved progress counters are invalid")
        rows.append({"arm_id": arm["id"], "saved_updates": completed, "planned_updates": plan["updates"],
                     "saved_failed_attempts": attempted-completed, "checkpoint": manifest.get("latest")})
        forecast += (plan["updates"]-completed)*float(times.get(arm["kind"], 0))
    return {"arms": rows, "remaining_training_hours": forecast/3600 if len(times)==2 else None,
            "scope": "Saved checkpoints only; up to 50 live updates may not yet appear. Time forecast excludes remaining preflight, evaluation and checkpoint I/O."}

def monitor_study(root, run_id, poll_seconds=10):
    from IPython.display import Markdown, display
    artifact = Path(root) / "artifacts/notebook_05" / run_id
    session = "fmm-scaling-" + run_id
    handle = display(Markdown("Connecting to the detached Notebook05 worker…"), display_id=True)
    missing_since = None
    while True:
        path = artifact / "study_status.json"
        status = _read(path) if path.exists() else {"status":"STARTING","stage":"launching"}
        stage = status.get("stage","")
        role = status.get("last_stage") or stage.split(":")[0]
        detail = ""
        folder = artifact / "stages" / role
        for name in (("evaluation_status.json",) if stage.endswith(":evaluation") else ("stage_status.json", "factorial_status.json")):
            if (folder / name).exists():
                progress = _read(folder / name)
                detail = "; " + json.dumps({k:v for k,v in progress.items() if k in ("arm_id","round_endpoint","completed_models","total_models","model_id","milestone")})
                break
        estimate_path = folder / "runtime_estimate.json"
        if estimate_path.exists():
            estimate = _read(estimate_path)
            detail += "; calibrated stage allowance: " + format(estimate["required_seconds"]/3600, ".2f") + "h at stage entry"
        progress = saved_training_progress(root, run_id, role)
        table = ""
        if progress["arms"]:
            rows = ["| Arm | Saved updates | Saved failed attempts |", "|---|---:|---:|"]
            for row in progress["arms"]:
                rows.append("| " + row["arm_id"] + " | " + str(row["saved_updates"]) + "/" + str(row["planned_updates"]) + " | " + str(row["saved_failed_attempts"]) + " |")
            table = "\n\n" + "\n".join(rows) + "\n\n" + progress["scope"]
            if progress["remaining_training_hours"] is not None and not stage.endswith(":evaluation"):
                table += " Remaining training forecast from calibration: " + format(progress["remaining_training_hours"], ".2f") + "h."
        if status.get("error"):
            table += "\n\n**Stopped:** " + status["error"] + " Saved checkpoints are retained; no failed attempt is silently skipped."
            if status.get("numerical_cause"):
                table += " Cause: " + status["numerical_cause"]
        clock = _read(artifact / "budget_clock.json")
        elapsed = (time.time()-clock["started_unix"])/3600
        remaining = max(0,clock["deadline_unix"]-time.time())/3600
        handle.update(Markdown("**" + status["status"] + "** — " + stage + detail + "\n\nElapsed: " +
                      format(elapsed,".2f") + "h; hard-limit time left: " + format(remaining,".2f") +
                      "h. Checkpoints and logs: \u0060" + str(artifact) + "\u0060." + table))
        if status["status"] == "COMPLETED":
            return status
        if status["status"] == "INCOMPLETE":
            return status
        if status["status"] == "FAILED":
            raise RuntimeError(status)
        if not _session_exists(session):
            missing_since = missing_since or time.time()
            if time.time()-missing_since > 30:
                raise RuntimeError("Worker exited without terminal evidence. Inspect worker.log; Run All resumes valid checkpoints while the original clock remains")
        else:
            missing_since = None
        time.sleep(min(30,max(1,poll_seconds)))

def display_results(root, run_id):
    from IPython.display import Markdown, Image, display
    artifact = Path(root) / "artifacts/notebook_05" / run_id
    path = artifact / "study_status.json"
    if not path.exists():
        display(Markdown("**NOT_RUN.** Run All starts the configured study."))
        return {"status":"NOT_RUN"}
    status = _read(path)
    display(Markdown("**Study outcome:** " + status.get("outcome",status["status"])))
    report = artifact / "combined_results.json"
    if report.exists():
        value = _read(report)
        for key in ("report_path","summary_path"):
            if value.get(key) and Path(value[key]).exists():
                display(Markdown(Path(value[key]).read_text()))
                break
    for folder in [artifact] + [artifact/"stages"/s["id"] for s in STAGES]:
        for png in sorted(folder.glob("*.png")):
            display(Image(filename=str(png)))
    return status
